***Cel analizy - klasteryzacja doroslych zdrowych mezczyzn w wieku 20 - 45 lat w celu znalezienia klastrow bedacych w grupie ryzyka do zachorowania na cukrzyce.*** 


In [2]:
import pandas as pd 

In [51]:
#import wszystkich datasetow i zebranie ich w jeden

df1 = pd.read_csv(r'C:/ML_Projects/ML_notebooks/NHANES_Clustering/NHANES_data\demographic.csv')
df2 = pd.read_csv(r'C:/ML_Projects/ML_notebooks/NHANES_Clustering/NHANES_data\questionnaire.csv')
df3 = pd.read_csv(r'C:/ML_Projects/ML_notebooks/NHANES_Clustering/NHANES_data\examination.csv')
df4 = pd.read_csv(r'C:/ML_Projects/ML_notebooks/NHANES_Clustering/NHANES_data\labs.csv')

#importowanie kolumn waznych w analizie ktore sa niedostepne w datasecie na kaggle
ghb = pd.read_sas(r'C:\ML_Projects\ML_notebooks\NHANES_Clustering\nhanes_data\GHB_H.xpt')
glu = pd.read_sas(r'C:\ML_Projects\ML_notebooks\NHANES_Clustering\nhanes_data\GLU_H.xpt')

In [52]:
# przygotowanie kolumn ktore beda potrzebne do analizy
df1 = df1[['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDSTATR' ,'RIDRETH3', 'INDFMPIR']]
df2 = df2[['SEQN', 'DIQ010']]
df3 = df3[['SEQN', 'BMXWAIST', 'BMXBMI', 'BPXSY1', 'BPXSY2', 'BPXSY3', 'BPXSY4', 'BPXDI1', 'BPXDI2', 'BPXDI3', 'BPXDI4']]
df4 = df4[['SEQN', 'LBXTR', 'LBDHDD', 'LBDLDL', 'LBXSATSI', 'LBXSUA']]
glu = glu[['SEQN', 'LBXGLU']]
ghb = ghb[['SEQN', 'LBXGH']]

In [53]:
df1 = df1.merge(df2, how='left', on='SEQN')
df1 = df1.merge(df3, how='left', on='SEQN')
df1 = df1.merge(df4, how='left', on='SEQN')
df1 = df1.merge(glu, how='left', on='SEQN')
df1 = df1.merge(ghb, how='left', on='SEQN')


In [54]:
df = df1.rename(columns={
    'SEQN': 'id',
    'RIAGENDR': 'gender',
    'RIDAGEYR': 'age',
    'RIDSTATR': 'test_status',
    'RIDRETH3': 'race',
    'INDFMPIR': 'poverty_ratio', #wskaznik ubostwa, <1 to uboga rodzina, >2 to x krotnosc rocznego dochodu
    'DIQ010': 'diabetes', # 1-tak, 2-nie, 3-stan krytyczny
    'BMXWAIST': 'waist_cm',
    'BMXBMI': 'bmi',
    **{f'BPXSY{i}': f'sys_{i}' for i in range(1, 5)}, #cisnienie skurczowe
    **{f'BPXDI{i}': f'dia_{i}' for i in range(1, 5)}, #cisnienie rozkurczowe
    'LBXTR': 'triglycerides',
    'LBDHDD': 'hdl',
    'LBDLDL': 'ldl',
    'LBXSATSI': 'alt',
    'LBXSUA': 'uric_aid', #kwas moczowy
    'LBXGLU': 'glucose',
    'LBXGH': 'hemoglobin'

})

In [55]:
# wybieranie jedynie pacjentow spelniajacych kryteria analityczne

men = df[(df['gender'] == 1) & (df['age'] <= 45) & (df['age'] > 20)]
men.head()

,id,gender,age,test_status,race,poverty_ratio,diabetes,waist_cm,bmi,sys_1,...,dia_2,dia_3,dia_4,triglycerides,hdl,ldl,alt,uric_aid,glucose,hemoglobin
8,73565,1,42,1,2,5.00,2.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,73577,1,32,2,1,0.29,2.0,100.0,28.9,118.0,...,74.0,78.0,NaN,148.0,33.0,119.0,44.0,6.1,104.0,5.3
28,73585,1,28,2,6,2.26,2.0,90.3,30.1,106.0,...,74.0,66.0,NaN,NaN,55.0,NaN,28.0,6.6,NaN,5.0
32,73589,1,35,2,3,1.74,2.0,94.6,26.2,124.0,...,66.0,72.0,NaN,NaN,45.0,NaN,21.0,6.8,NaN,5.5
35,73592,1,29,2,1,2.14,2.0,107.7,31.0,118.0,...,78.0,76.0,NaN,NaN,34.0,NaN,49.0,8.8,NaN,5.7


In [56]:
men.shape

(1188, 24)

Niepotrzebne sa dane jak sys_1, sys_2, dia_1, dia_2 itp <br>
Najlepiej policzyc jest srednia z tych wartosci i na takich danych pracowac.

In [57]:
men['sys'] = men[['sys_1', 'sys_2', 'sys_3', 'sys_4']].mean(axis=1)
men['dia'] = men[['dia_1', 'dia_2', 'dia_3', 'dia_4']].mean(axis=1)

men = men.drop(columns=['sys_1', 'sys_2', 'sys_3', 'sys_4', 'dia_1', 'dia_2', 'dia_3', 'dia_4'])

C:\Users\norbe\AppData\Local\Temp\ipykernel_17636\1786704316.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  men['sys'] = men[['sys_1', 'sys_2', 'sys_3', 'sys_4']].mean(axis=1)
C:\Users\norbe\AppData\Local\Temp\ipykernel_17636\1786704316.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  men['dia'] = men[['dia_1', 'dia_2', 'dia_3', 'dia_4']].mean(axis=1)


In [60]:
# zajrzenie w dane
men.describe()

,id,gender,age,test_status,race,poverty_ratio,diabetes,waist_cm,bmi,triglycerides,hdl,ldl,alt,uric_aid,glucose,hemoglobin,sys,dia
count,1188.000000,1188.0,1188.000000,1188.000000,1188.000000,1097.000000,1188.000000,1114.000000,1138.000000,503.000000,1094.000000,484.000000,1093.000000,1093.000000,504.000000,1101.000000,1109.000000,1109.000000
mean,78743.378788,1.0,33.021886,1.963805,3.389731,2.476582,1.985690,97.714632,28.470211,139.990060,46.696527,114.429752,32.387923,6.041629,102.007937,5.420163,119.758341,70.753832
std,2885.552999,0.0,7.254399,0.186854,1.619842,1.652196,0.206786,16.414304,6.729617,149.294536,13.073977,34.966195,23.911687,1.238602,26.984937,0.725530,12.586807,11.904589
min,73565.000000,1.0,21.000000,1.000000,1.000000,0.000000,1.000000,55.500000,14.100000,20.000000,10.000000,42.000000,7.000000,2.000000,61.000000,3.800000,82.000000,0.000000
25%,76345.250000,1.0,27.000000,2.000000,3.000000,1.040000,2.000000,85.725000,23.900000,65.000000,38.000000,88.000000,20.000000,5.200000,92.000000,5.100000,111.333333,64.000000
50%,78726.500000,1.0,33.000000,2.000000,3.000000,2.120000,2.000000,95.700000,27.300000,98.000000,45.000000,112.000000,26.000000,6.000000,98.000000,5.300000,118.000000,71.333333
75%,81192.000000,1.0,39.000000,2.000000,4.000000,4.100000,2.000000,106.900000,31.600000,157.000000,54.000000,138.000000,37.000000,6.800000,104.000000,5.600000,126.666667,77.333333
max,83727.000000,1.0,45.000000,2.000000,7.000000,5.000000,3.000000,162.800000,67.900000,1637.000000,128.000000,244.000000,317.000000,13.300000,368.000000,11.500000,193.333333,128.000000


In [62]:
#sprawdzenie ktore kolumny maja malo danych

for col in men.columns:
    if men[col].notna().sum() < 800:
        print(f'{col}: {men[col].notna().sum()}')

triglycerides: 503
ldl: 484
glucose: 504


Glukoza jest cecha z ktora bedzie sprawdzana jakosc klasteryzacji dlatego teraz jest ona nieistotna. <br>
LDL zas nie wnosi nic ponad TG i HDL dlatego jest jeszcze mniej istotnie niz glukoza. <br>
Triglicerydy zas sa bardzo waznym elementem - ich ilosc w organizmie mowi o problemach ukladu krazenia.


Skoro dataset posiada okolo 500 rekordow tg mozna ta kolumne wlaczyc do analizy.

!! wybrac filtracje tg lub imputowanie tg oraz wyjasnic dlaczego!! <br>
nie imputowac tylko filtrowac bo mediana bardzo by zepsula dane (za duzo nan zeby imputowac cokolwiek)